# Reduciendo la Redundancia de Sensores en Línea de Producción con PROC GVARCLUS


## Resumen Ejecutivo

Una línea de fabricación multi-zona transmite decenas de lecturas de sensores correlacionadas, muchas de las cuales llevan la misma señal subyacente. Este notebook usa **PROC GVARCLUS** (agrupamiento de variables) para agrupar los 13 sensores de proceso en cuatro clústeres disjuntos, luego lee los valores R-cuadrado de cada clúster para elegir un medidor representativo por clúster — reduciendo la huella de monitoreo SPC sin perder información de proceso. Tres de los cuatro clústeres se mapean limpiamente sobre las zonas físicas (R-cuadrado promedio 0.92, 0.93 y 0.96); el cuarto es un clúster de un solo canal que el procedimiento separó, el cual examinamos en lugar de pasarlo por alto.


## Fuentes de Datos

Todos los datos se generan en línea con `call streaminit(20260531)` y `rand()` — sin entradas externas ni de red.

| Conjunto de Datos | Filas | Variables Clave | Descripción |
|---------|------|---------------|-------------|
| `process_sensors` | 100 | `zone1_a`–`zone1_c`, `zone2_a`–`zone2_c`, `zone3_a`, `zone3_b`, `temp_in`, `temp_out`, `pressure_a`, `pressure_b`, `vibration` | Lecturas sintéticas de una línea de producción de 3 zonas. Los sensores dentro de una zona comparten un impulsor latente (alta correlación); se agregan medidores de zonas cruzadas (temperaturas ligadas a la zona 1, presiones a la zona 2, vibración a la zona 3) para crear redundancia realista. El bucle del DATA step solicita 300 ciclos, pero este entorno de evaluación se ejecuta en modo sin licencia y retiene las primeras 100 observaciones — suficiente para recuperar la estructura de clústeres limpiamente. |
| `clusters` (OUT=) | 13 | `Variable`, `Cluster`, `RSq_Own` | Una fila por sensor de entrada: el clúster al que fue asignado y su R-cuadrado con el componente de ese clúster. Producido por la sentencia `OUTPUT OUT=`. |


# Reduciendo la Redundancia de Sensores en Línea de Producción con PROC GVARCLUS

En una línea de producción instrumentada, es común registrar decenas de sensores que miden fenómenos físicos superpuestos — múltiples termopares en una zona, transductores de presión redundantes, sondas de vibración que rastrean el mismo eje. Alimentar cada canal a una carta de control o a un modelo posterior desperdicia esfuerzo de monitoreo e infla la multicolinealidad.

**El agrupamiento de variables** aborda esto directamente. `PROC GVARCLUS` particiona las variables numéricas en clústeres *disjuntos*, donde cada clúster se resume mediante el primer componente principal de sus miembros. Los sensores que se mueven juntos caen en el mismo clúster; el ingeniero puede entonces mantener un canal representativo por clúster.

En este notebook nosotros:

1. Sintetizamos lecturas de una línea de 3 zonas con sensores deliberadamente redundantes (el bucle solicita 300 ciclos; se retienen 100 aquí).
2. Agrupamos los 13 canales con `PROC GVARCLUS` con `MAXCLUSTERS=4`.
3. Capturamos las asignaciones de clúster en un conjunto de datos de salida y las resumimos.
4. Interpretamos los valores R-cuadrado para elegir un medidor por clúster para el SPC continuo.


## Paso 1 — Generar datos sintéticos de sensores

Simulamos tres zonas de proceso, cada una con un impulsor oculto (`zone1_a`, `zone2_a`, `zone3_a`). Los canales restantes son funciones lineales ruidosas del impulsor de su zona, de modo que están fuertemente correlacionados dentro de una zona pero casi independientes entre zonas. También ligamos las temperaturas de entrada/salida a la zona 1 y los dos transductores de presión a la zona 2, imitando la redundancia entre instrumentos observada en líneas reales. Una semilla fija hace que la ejecución sea reproducible. El bucle solicita 300 ciclos; en modo sin licencia el motor retiene las primeras 100 observaciones, lo cual la NOTA a continuación reporta.


In [1]:
DATOS process_sensors;
    LLAMAR streaminit(20260531);
    HACER cycle = 1 HASTA 300;
        /* Zona 1: impulsor latente mas dos sondas redundantes */
        zone1_a = rand("normal", 0, 1);
        zone1_b = 0.90*zone1_a + rand("normal", 0, 0.30);
        zone1_c = 0.85*zone1_a + rand("normal", 0, 0.35);

        /* Zona 2: impulsor latente mas dos sondas redundantes */
        zone2_a = rand("normal", 0, 1);
        zone2_b = 0.88*zone2_a + rand("normal", 0, 0.30);
        zone2_c = 0.82*zone2_a + rand("normal", 0, 0.40);

        /* Zona 3: impulsor latente mas una sonda redundante */
        zone3_a = rand("normal", 0, 1);
        zone3_b = 0.90*zone3_a + rand("normal", 0, 0.30);

        /* Canales de instrumentos cruzados ligados a impulsores existentes */
        temp_in    = 180 + 4.0*zone1_a + rand("normal", 0, 1.5);
        temp_out   = 178 + 4.0*zone1_a + rand("normal", 0, 1.6);
        pressure_a = 2.5 + 0.60*zone2_a + rand("normal", 0, 0.20);
        pressure_b = 2.5 + 0.58*zone2_a + rand("normal", 0, 0.22);
        vibration  = 0.40 + 0.30*zone3_a + rand("normal", 0, 0.10);
        SALIDA;
    END;
    ELIMINAR cycle;
EJECUTAR;



NOTE: DATA process_sensors

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote process_sensors (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.06 seconds
  cpu   0.06 seconds


## Paso 2 — Agrupar los sensores

Llamamos a `PROC GVARCLUS` sobre los 13 canales. La sentencia `VAR` lista los sensores numéricos a agrupar. Limitamos la partición a cuatro clústeres con `MAXCLUSTERS=4` (esperamos aproximadamente un clúster por zona física, con algo de holgura). `ODS GRAPHICS ON` con `PLOTS=ALL` habilita el dendrograma de clúster de variables; el motor escribe ese SVG al directorio de trabajo en lugar de renderizarlo en línea, así que la estructura que leemos a continuación proviene de la tabla impresa **Variable Cluster Assignments** y de la tabla de valores propios por clúster.

La sentencia `OUTPUT OUT=` escribe las asignaciones de variable a clúster — junto con el R-cuadrado de cada variable respecto a su propio clúster — a un conjunto de datos con el que podemos programar más adelante.


In [2]:
ODS GRAPHICS ON;

PROCEDIMIENTO gvarclus DATOS=process_sensors maxclusters=4 PLOTS=ALL;
    VAR zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
    SALIDA out=clusters;
EJECUTAR;

ODS GRAPHICS OFF;



                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: ODS Graphics is ON (width=640px, height=480px, format=SVG).
NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.
NOTE: ODS Graphics is OFF.


## Paso 3 — Reejecutar con la opción SUMMARY

Ejecutar el procedimiento una segunda vez con la opción `SUMMARY` mantiene la misma partición. `PLOTS=NONE` desactiva los gráficos en esta pasada. En este entorno el reporte impreso es idéntico al Paso 2 — la misma tabla de asignación de 13 filas, los mismos cuatro clústeres y el mismo resumen de valores propios — así que esta celda principalmente demuestra que las opciones `SUMMARY` y `PLOTS=NONE` se analizan y ejecutan correctamente; no se espera que agregue números nuevos.


In [3]:
PROCEDIMIENTO gvarclus DATOS=process_sensors maxclusters=4 summary PLOTS=none;
    VAR zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
EJECUTAR;



                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.


## Paso 4 — Inspeccionar el conjunto de datos de salida

El conjunto de datos `clusters` de `OUTPUT OUT=` tiene una fila por sensor con su `Cluster` asignado y `RSq_Own` (la correlación al cuadrado entre el sensor y el componente de su clúster). Un `RSq_Own` alto significa que el sensor está bien representado por su clúster — un candidato ideal para *descartar* en favor del representante del clúster. Ordenamos por clúster y luego por R-cuadrado, de modo que el sensor más representativo de cada clúster quede en la parte superior de su grupo.


In [4]:
PROCEDIMIENTO ORDENAR DATOS=clusters out=clusters_ranked;
    POR CLUSTER DESCENDENTE RSq_Own;
EJECUTAR;

PROCEDIMIENTO IMPRIMIR DATOS=clusters_ranked ETIQUETA;
    VAR Variable CLUSTER RSq_Own;
    ETIQUETA Variable = "Canal del Sensor"
          CLUSTER  = "Clúster"
          RSq_Own  = "R-Cuadrado con su Propio Clúster";
EJECUTAR;



  Obs  Canal del Sensor   Clúster   R-Cuadrado con su Propio Clúster
-----  ----------------  --------  ---------------------------------
    1  ZONE1_A                  1                            0.96867
    2  ZONE1_B                  1                             0.9189
    3  TEMP_IN                  1                           0.903394
    4  TEMP_OUT                 1                           0.889519
    5  ZONE2_A                  2                            0.98364
    6  ZONE2_B                  2                           0.946708
    7  PRESSURE_A               2                           0.929356
    8  PRESSURE_B               2                           0.920915
    9  ZONE2_C                  2                           0.892405
   10  ZONE3_A                  3                           0.977237
   11  VIBRATION                3                            0.95916
   12  ZONE3_B                  3                           0.949054
   13  ZONE1_C                  4


NOTE: PROC SORT data=clusters

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 13 rows from clusters.
NOTE: Wrote clusters_ranked (13 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=clusters_ranked

NOTE: PROC PRINT completed: 13 observations printed, 3 variables


## Interpretando los resultados

La partición recupera la mayor parte de la estructura física de la línea, con una advertencia honesta:

- **Clúster 1** agrupa `zone1_a` (R²=0.969), `zone1_b` (0.919) y las temperaturas de entrada/salida `temp_in` (0.903) y `temp_out` (0.890) — cuatro de los cinco canales impulsados por la señal latente de zona 1. R-cuadrado promedio 0.920.
- **Clúster 2** agrupa los tres canales de zona 2 `zone2_a` (0.984), `zone2_b` (0.947), `zone2_c` (0.892) junto con los dos transductores de presión `pressure_a` (0.929) y `pressure_b` (0.921), que ligamos al impulsor de zona 2. R-cuadrado promedio 0.935.
- **Clúster 3** agrupa `zone3_a` (0.977), `zone3_b` (0.949) y `vibration` (0.959). R-cuadrado promedio 0.962.
- **Clúster 4** es un solo canal: `zone1_c`, la tercera sonda de zona 1, fue separado por sí solo con R²=1.000 (un singleton es, trivialmente, explicado a la perfección por sí mismo). A pesar de compartir el impulsor de zona 1, con este tamaño de muestra el procedimiento juzgó que `zone1_c` era lo suficientemente distinto del grupo `zone1_a`/temperaturas como para justificar su propio clúster en lugar de fusionarlo con el Clúster 1.

Los tres clústeres multi-canal tienen cada uno un R-cuadrado promedio superior a **0.90**, confirmando que un solo componente explica la mayor parte de la variación entre sus miembros — exactamente la redundancia que queremos colapsar. El valor atípico solitario — `zone1_c` cayendo en su propio clúster en lugar de con el resto de la zona 1 — es un recordatorio útil de que el agrupamiento de variables está guiado por los datos: con más ciclos o un presupuesto de `MAXCLUSTERS` más alto, el límite puede cambiar, así que la partición es un punto de partida para el juicio de ingeniería, no una verdad fija.

**Acción para la línea.** Dentro de cada clúster multi-canal, mantener el sensor con el `RSq_Own` más alto (el canal más representativo de su clúster) y retirar o de-priorizar el resto del monitoreo SPC de rutina — por ejemplo `zone2_a` para el Clúster 2 y `zone3_a` para el Clúster 3. Tratar el singleton `zone1_c` como una bandera para investigar en lugar de una retención automática: confirmar si lleva información genuinamente distinta o es un artefacto del agrupamiento antes de decidir monitorearlo por separado. Incluso con ese canal aparte, esto colapsa 13 canales monitoreados hacia un plan de monitoreo de cuatro canales, reduciendo el ruido de alarmas y la multicolinealidad mientras se preserva el contenido de información de la línea.
